### Auto-Generated Code Cells

The following code cells contain code from external .py files that has been automatically added to remove these dependencies. This allows you to run this version of the SELENE notebook on Cloud platforms such as Google Colab. Run all following code cells before starting with the actual notebook content. 

Most Cloud platforms such as Google Colab have a wide range of Python packages preinstalled. However, if any import statements in the following code cells throws an error because of a missing package, you can create and run a new code cell to install the missing package using the `pip` command as shown here:
```
!pip install <package-name>
```

You can use the code cell below for convenience.

In [ ]:
#!pip install <package-name>

In [ ]:
# === Inline content for module src.utils.data.files  ==

# === Inlined imports ===
import bz2
import zipfile, tarfile
from tqdm import tqdm
import os, re, yaml
import requests

# === Embedded configs ===
CONFIG = yaml.safe_load("""urls:
  downloads:
    notebooks: "https://github.com/chrisvdweth/selene/tree/master/notebooks"
    datasets: "https://chrisvdw.net/projects/selene/downloads/datasets/"
    models: "https://chrisvdw.net/projects/selene/downloads/models/"
    """)

# === Inlined definitions ===
def create_folder(folder_name, exist_ok=True):
    try:
        os.makedirs(folder_name, exist_ok=exist_ok)
        return folder_name
    except:
        return None

In [ ]:
# === Inline content for module src.utils.compute.gpu  ==

# === Inlined imports ===
import torch
import random

# === Embedded configs ===


# === Inlined definitions ===
def select_device(force_cpu=False, required_memory_gb=None):
    if force_cpu == True:
        return torch.device('cpu')
    candidate_gpus = []
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(device=i)
        if required_memory_gb is None or free / (1024 * 1024 * 1024) > required_memory_gb:
            candidate_gpus.append((i, free))
    if len(candidate_gpus) == 0:
        return torch.device('cpu')
    else:
        device = torch.cuda.device(random.choice(candidate_gpus)[0])
        return torch.device(f'cuda:{device.idx}')

# Training BERT from Scratch

This notebook walks through the core steps of training a **BERT* model from scratch: preparing the training data, defining the model architecture, and running the actual training loop. Pre-trained BERT models are usually just downloaded and used as a black box, which hides most of what happens during pre-training. Here we open that box. We turn a raw text corpus into sentence pairs for **Next Sentence Prediction (NSP)**, apply the 80/10/10 corruption for **Masked Language Modeling (MLM)**, and build the three input representations (token, segment, and position IDs) together with the attention mask and the MLM and NSP labels. We then implement the model itself, covering the embeddings, the encoder stack, the pooler, the two pre-training heads, and weight tying, and finally train it with a suitable loss, optimizer, learning rate schedule, and validation.

The purpose of this notebook is explicitly *not* to obtain state-of-the-art results or well-learned embeddings. Doing so would require far more data, a much larger model, and weeks of training on dedicated hardware. Instead, we deliberately use a small dataset (WikiText) and a tiny model configuration that trains in minutes, so that the complete pipeline can be run and inspected end to end. The goal is to understand and comprehend the details that training requires, and to verify that every component works together: if the training and validation losses decrease and the NSP accuracy increases, the pipeline is doing what it should. To connect this to practice, we also show how the same steps can be "speed-run" with the `transformers` library, which is the more practical route but hides many of these details.

This notebook assumes that you are already familiar with BERT. In particular, you should know its Transformer-based, encoder-only architecture and the two original learning objectives, MLM and NSP, including why they are used as proxy tasks to obtain contextual embeddings. We do not re-explain these concepts here, and instead focus on how they are turned into concrete data, code, and training steps.

### Setting up the Notebook

#### Make Required Imports

This notebook requires the import of different Python packages but also additional Python modules that are part of the repository. If a package is missing, use your preferred package manager (e.g., [conda](https://anaconda.org/anaconda/conda) or [pip](https://pypi.org/project/pip/)) to install it. If the code cell below runs with any errors, all required packages and modules have successfully been imported.

In [1]:
import re
import gc
import time
import numpy as np

from transformers import (
    AutoTokenizer,
    BertConfig, 
    BertForPreTraining
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from dataclasses import dataclass


#### Create Output Folder

When training a BERT model, we periodically save the current state of the model to disk. Although the dataset and model used in this notebook is relatively small, we still adopt this best practice of saving the model to avoid losing the training progress in case of any errors or crashes. You can change the output folder by editing the code cell below.

In [2]:
output_folder = create_folder("data/generated/models/bert/")

print(f"Output folder: {output_folder}")

Output folder: data/generated/models/bert/


#### Checking & Setting Computing Device

PyTorch allows to train neural networks on supported GPU to significantly speed up the training process. If you have a support GPU, feel free to utilize it. However, for this notebook it's certainly not needed as our dataset is small and our network model is very simple. We provide an auxiliary method to automatically select the best device. It checks if a supported GPU is available and if so, uses it as the preferred device.

In [3]:
# Select preferred device (GPU, if available; CPU otherwise); you can enfore the use of the CPU
DEVICE = select_device(force_cpu=False)

print("Available device: {}".format(DEVICE))

Available device: cuda:0


#### Preliminaries

While there are brief recap, this notebook assumes a basic understanding of
* Word embeddings in general (i.e., the overall purpose and alternative methods such as Word2Vec or ELMo)
* The transformer architecture, including the attention mechanisms, positional encodings, as well as attention and loss masking

---

## BERT in a Nutshell

### Goal: Contextual Word Embeddings

BERT's goal is to learn **contextual word embeddings**: instead of assigning each word one fixed vector, it computes a representation for every token that depends on the entire sentence. The word *"bank"* therefore gets different vectors in *"river bank"* and *"bank account"*, which lets the model handle ambiguity, syntax, and long-range dependencies far better than static embeddings such as Word2Vec. These embeddings are so useful because they are learned from huge amounts of unlabeled text through the proxy tasks MLM and NSP &mdash; see below &mdash; and can then be reused for many tasks, either as features for other models or by fine-tuning the pretrained encoder with a small task-specific head on limited labeled data.

Architecturally, BERT is based on an **encoder-only Transformer**, whose self-attention lets every token attend directly to every other token in the sequence, in both directions. This typically outperforms earlier contextual models such as ELMo, which relies on recurrent networks (LSTMs). Those process text sequentially, are harder to parallelize, and struggle more with long-range dependencies. Moreover, ELMo only concatenates separately trained left-to-right and right-to-left representations, whereas BERT's encoder conditions on both sides jointly in every layer. The result is richer contextual embeddings and, in practice, stronger performance across a wide range of NLP tasks.

### Learning Objectives

#### Masked Language Model (MLM)

The MLM learning objective directly implements the **Distributional Hypothesis** which says that a word's meaning is determined by the contexts in which it appears. MLM implements this almost literally: a word is hidden, and the model must predict it using only the surrounding words, left and right. To do this well across billions of tokens, the network has to learn representations in which words that occur in similar contexts are treated similarly, since they must lead to similar predictions. More specifically, during training, a fixed proportion of input tokens (typically around 15%) is selected at random and their original identities are hidden from the model. Of those 15% selected tokens:
  * 80% are replaced with a `[MASK]` token,
  * 10% are replaced with a random token,
  * 10% are left unchanged.

Only, those ~15% of selected tokens the model considered when computing the loss during training. To show an example, consider the following simple sentence:

> I watched a movie last night on an old DVD I found about a man who lost his memory.

As output, we might get the following corrupted sequence:

> I watched a <font color="red"><b>[MASK]</b></font> last night on an old <font color="red"><b>river</b></font> I found about a <font color="red"><b>[MASK]</b></font> who lost his <font color="red"><b>memory</b></font>.

Here, a total of four tokens have been selected. Both *"watched"* and *"man"* have been masked (i.e., replaced with the `[MASK]` token). *"DVD"* has been replaced with the random token *"banana"*, and *"memory"* has been kept as is. The `[MASK]` token appears only during pre-training and never in downstream tasks, so if every selected token were masked, the model would only learn good representations at `[MASK]` positions. Replacing some selected tokens with a random word or leaving them unchanged fixes this: since the model cannot tell which visible tokens are targets or corrupted, it must build a context-aware representation for every token and check each one against its context. This reduces the mismatch between pre-training and fine-tuning, where all inputs are real tokens.

#### Next Sentence Prediction (NSP):

A training sample for BERT contains pairs of sentences. In ~50% of the cases, the second sentence truly follows the first one in the original corpus; in the other 50%, it is a randomly sampled sentence from the corpus. The two sentences are concatenated into a single input sequence, separated by a special `[SEP]` token, and a classification token `[CLS]` is added at the beginning. The model is then trained to predict whether the second sentence is the actual continuation of the first. The goal of NSP is to encourage BERT to capture inter-sentential coherence and discourse-level information, which is important for tasks involving sentence pairs, such as question answering and natural language inference. By framing this as a binary classification problem over the `[CLS]` representation, BERT learns to encode not only the meaning of individual sentences but also how sentences relate to each other in context. 

The figure below illustrates the overall training setup of BERT with both learning objectives. This example assumes that this sentence pair does indeed represent two adjacent sentences in the same document, and the binary class label for the NSP is $0$; for unrelated sentences the label is $1$ (i.e., "is random"). Similar to the previous example, a total of four tokens have been selected for the MLM learning objective, two are masked, one is replaced with a random token, and one is left unchanged.

<img src="https://raw.githubusercontent.com/chrisvdweth/selene/refs/heads/master/notebooks/images/illustrations/bert/bert-example-setup.png" style="margin:auto;max-width:900px;width:100%" alt="BERT - Example Setup" />

<br />

As we will see in a moment, a significant part of the implementation is to generate valid training samples given a large corpus of text documents.

---

## Data Preparation

### Downloading Corpus


* **WikiText-2** (`wikitext-2-raw-v1`) is a small language modeling dataset of over 2 million tokens extracted from a set of verified *Good* and *Featured* articles on Wikipedia. It is the "raw" version of the WikiText-2 dataset, meaning it retains original punctuation, case, numbers, and formatting tokens without low-frequency words being mapped to specialized UNK tokens. Because it preserves full article structures and handles long-term context dependencies better than older benchmarks like the Penn Treebank, it is widely used as a standard benchmark for evaluating language models, natural language processing tasks, and character- or subword-level machine learning architectures.

* **WikiText-103** (`wikitext-103-raw-v1`) is the larger sibling of WikiText-2 It contains more than 103 million training tokens (about 28,000 articles), while the validation and test splits are the same as in WikiText-2. It has long been a standard benchmark for language modeling, and it is a more realistic corpus for training a small BERT model from scratch, although still tiny compared with the roughly 3.3B words used for the original BERT. It takes a few hundred MB of disk space, so loading and tokenizing it takes noticeably longer than for WikiText-2.

Thus, by default, we will be using `wikitext-2-raw-v1` &mdash; however, you are welcome to change the code cell below to work with `wikitext-103-raw-v1`. Both datasets are easily available in the Hugging Face `datasets` library, where each row is a single line of text, with many empty lines and section headings that usually need to be filtered out; something we therefore have to do as well.

In [4]:
corpus = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
#corpus = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

In [5]:
print(f"Number of training samples:   {corpus['train'].shape[0]}")
print(f"Number of validation samples: {corpus['validation'].shape[0]}")
print(f"Number of test samples:       {corpus['test'].shape[0]}")

Number of training samples:   36718
Number of validation samples: 3760
Number of test samples:       4358


### Cleaning & Vectorization

The raw WikiText files are not yet in a form BERT can consume. Each row is a single line of text, many rows are empty or contain only section headings (e.g. `= Title =`), and some lines contain formatting artifacts. Training on these would waste compute on empty sequences and add noise. The first step is therefore to clean the data by removing empty lines and headings, and, because NSP and sentence-pair construction need documents with contiguous text, to keep track of which lines belong together.

The cleaned text must then be converted into token IDs, since the model only accepts integers that index its embedding tables. The matching pretrained tokenizer splits each line into WordPiece tokens and maps them to their vocabulary IDs. Doing this once up front, rather than during every training step, makes all later steps (pairing segments, adding `[CLS]`/`[SEP]`, masking) fast and simple, because they can operate directly on lists of integers.

#### Cleaning

To better understand the need for cleaning, let's just look at some of the first lines of the training sets.

In [6]:
for idx, line in enumerate(corpus["train"]["text"][:10]):
    print(line)


 = Valkyria Chronicles III = 


 Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . 

 The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series 

If you look at the output, you will notice uncommon patterns using the `@` characters, mainly `@-@`, `@,@`, and `@.@`. These patterns are tokenizing artifacts representing the hyphen  in hyphenated words as well as the comma and period punctuation marks. This is in fact very convenient since these three characters are often used in other contexts; for example, the period character is also used as a decimal point. For pretraining BERT, however, we have to reverse the process which is straightforward implement as shown by the `clean()` method in the code cell bellow

In [7]:
def clean(s): 
    return s.replace(" @-@ ", "-").replace(" @,@ ", ",").replace(" @.@ ", ".")

To see the method's effect, let's again print the first few lines but now `clean()` is first applied to each line.

In [8]:
for idx, line in enumerate(corpus["train"]["text"][:10]):
    print(clean(line))


 = Valkyria Chronicles III = 


 Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role-playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real-time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . 

 The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomer

Of course, we still have empty lines and section headings (e.g. `= Title =`). But these we can easily filter later when iterating through all articles and all lines within each article.

#### Vectorization

Like most modern language models, BERT relies on a pretrained **subword tokenizer** (WordPiece) rather than on whole words or single characters. Whole-word vocabularies are huge and still miss rare words, misspellings, and new names, which become out-of-vocabulary tokens. Characters avoid this problem but produce very long sequences and carry little meaning per token. Subword tokenization strikes a balance: frequent words stay whole, while rare words are split into smaller, reusable pieces (e.g. `un`, `##believ`, `##able`). This keeps the vocabulary small (about 30k entries), covers any input text, and keeps sequences reasonably short.

The tokenizer is also tightly coupled to the model, because the embedding table is indexed by the IDs of its specific vocabulary. A model can therefore only be used with the tokenizer it was pretrained with. In this notebook, we use the standard `bert-base-uncased` tokenizer. Training our own subword tokenizer (and then pre-training a model on it) would go well beyond the scope here, and the pretrained one is a solid, widely used default for English text.

The `AutoTokenizer` class from Hugging Face `transformers` loads the correct tokenizer for a given model name without you having to know the specific tokenizer class. Its `from_pretrained()` method downloads (and caches) the tokenizer's vocabulary and configuration from the Hugging Face Hub. For `bert-base-uncased`, this returns a fast WordPiece tokenizer (`BertTokenizerFast`) with the vocabulary and special tokens the model was pretrained with. Because the same code works for other model families (e.g. RoBERTa or DistilBERT) by only changing the model name, it keeps notebooks flexible and reduces the risk of pairing a model with the wrong tokenizer.

In [9]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(f"Size of vocabulary: {tokenizer.vocab_size}")

Size of vocabulary: 30522


The `bert-base-uncased` tokenizer automatically lowercases the text (and strips accents) as part of its preprocessing, so *"Paris"* and *"paris"* map to the same token. We therefore do not need to lowercase sentences or documents ourselves before tokenization. This also guarantees that the text is preprocessed exactly as it was during pre-training, which avoids mismatches that could silently degrade the embeddings.

With the tokenizer available, we can now convert a document (i.e., any string) into its corresponding sequence of token IDs. To this end, the code cell below implements the `doc_to_sentence_ids` method. It first cleans each paragraph with the helper method `clean()` and splits it into sentences using a simple regular expression that breaks after `.`, `!`, or `?` followed by whitespace, dropping empty strings. The pretrained tokenizer then converts all sentences in one batched call into WordPiece token IDs, with `add_special_tokens=False` so that no `[CLS]` or `[SEP]` is added yet, since these are inserted later when sentence pairs are assembled. Finally, each non-empty ID list is stored as a compact NumPy `int32` array.

**Important:** The regex splitter is only a heuristic: it can wrongly split after abbreviations such as "Dr." or "U.S.", but this is usually acceptable for a demonstration.

In [10]:
def doc_to_sentence_ids(doc, tokenizer):
    # Split document (here: an article) into its sentences
    sents = [s for p in doc for s in re.split(r"(?<=[.!?])\s+", clean(p)) if s]
    # Use pretrained tokenizer to convert tokens into their repective token IDs
    ids = tokenizer(sents, add_special_tokens=False)["input_ids"]
    # Return list of ID lists ()
    return [np.asarray(x, dtype=np.int32) for x in ids if x]

We can test the method by passing it a couple of example sentences together with the tokenizer. Notice the examples below assume the preprocessed format of the cleaned Wikitext dataset particularly that punctuation marks are enclosed by whitespaces not only after but also before the punctuation marks; otherwise our simple regex approach of splitting sentences would no longer work.

In [11]:
doc_to_sentence_ids(["What is the capital of France . The capital is Paris ."], tokenizer)

[array([2054, 2003, 1996, 3007, 1997, 2605, 1012], dtype=int32),
 array([1996, 3007, 2003, 3000, 1012], dtype=int32)]

We can also create the example from the figure above, i.e., the training sample of a pair of sentences including the special tokens. Again, this shows the importance that these special tokens must be known to the tokenizer. In other words, for example, we could not simple use, say, `[SPLIT]` instead of `[SEP]` since this token is unlikely to be in the tokenizer's vocabulary &mdash; and even if it would be a known token, it would not be associated with the correct token ID.

In [12]:
samples = ["[CLS] I [MASK] a movie last banana [SEP] The special [MASK] were all so cool [SEP] [PAD] [PAD]"]

doc_to_sentence_ids(samples, tokenizer)

[array([  101,  1045,   103,  1037,  3185,  2197, 15212,   102,  1996,
         2569,   103,  2020,  2035,  2061,  4658,   102,     0,     0],
       dtype=int32)]

This output shows then when using the `bert-base-uncased` tokenizer, the IDs for the special tokens are: `[CLS]` = $101$, `[SEP]` = $102$, `[MASK]` = $103$, `[PAD]` = $0$.

So far, we assume that we already have sentences or short paragraphs. However, we have already seen above that each document in the training datasets consists of multiple articles, including headings and empty lines we later want to ignore. To better see this, the code cell below mimics the content of a document using a tiny toy example. Note the `split()` method at the end that splits the strings line by line, just like a document is represented in the dataset.

In [13]:
demo_doc = """

= France =

This is an article about France .

= = History = = 

Let 's talk about the history of France .


= Paris =

This is an articel about Paris .

""".split("\n")

An obvious and simple goal is to ignore all empty lines. The heading lines, however, require some additional considerations. Overall, we want to ignore those, but the title headings (e.g., )`= France =` tell us where a new article starts. We want to separate articles from each other so we can later form pairs of sentences across different articles to form the negative samples (Class $0$) for the NSP learning objective. Thus, we can define the following regex pattern to match only title headings but any section or subsection headings.

In [14]:
TITLE = re.compile(r"= [^=].*[^=] =")

The `split_documents()` method in the code cell below now splits a training document with respect to the three main cases: (a) ignore empty lines (b) start a new paragraph if the current line is a title heading, (c) ignore all headings that are not title headings. The lines of the current paragraph is stored in variable `current`; variable `docs` contains all paragraph which is returned at the end.

In [15]:
def split_documents(lines):
    docs, current = [], []
    for line in lines:
        s = line.strip()
        if not s: continue                   # Ignore empty lines
        if TITLE.fullmatch(s):               # New article detected
            if current: docs.append(current) # Add current paragraph to docs
            current = []                     # Initialize new paragraph
        elif s.startswith("="): continue     # "= = Section = =" -> skip
        else:
            current.append(s)                # Add line to current paragraph
    if current: docs.append(current)         # Add last paragraph to docs
    return docs

To see how the method works, let's apply it to our small demo document.

In [16]:
docs = split_documents(demo_doc)

print(docs)

[['This is an article about France .', "Let 's talk about the history of France ."], ['This is an articel about Paris .']]


We can see that the document correctly split the two articles and otherwise ignored all empty lines and headings. The last auxiliary method is `build_docs()`, see below, which uses methods `split_documents()` and `doc_to_sentence_ids` to clean, split, and vectorize all documents in the datasets.

In [17]:
def build_docs(corpus, tokenizer, split="train"):
    docs = split_documents(corpus[split]["text"])
    
    return [d for d in map(lambda doc: doc_to_sentence_ids(doc, tokenizer), docs) if len(d) > 0]

This means we can finally vectorize all documents of our dataset.

In [18]:
train_docs = build_docs(corpus, tokenizer, "train")
val_docs   = build_docs(corpus, tokenizer, "validation")
test_docs  = build_docs(corpus, tokenizer, "test")

print(f"Number of articles (training/validation/testing): {len(train_docs)}/{len(val_docs)}/{len(test_docs)}")

Number of articles (training/validation/testing): 629/60/62


Although we now have successfully vectorized the dataset, each article is represented by a list of sentences, with each sentence represented as a sequence of token IDs. However, for training a BERT model, particularly for the NSP learning objective, we have to generate both types of sentences: positive pairs (i.e., the two sentences are adjacent in the same document $\Rightarrow$ NSP class $1$) and negative pairs (i.e., the two sentences are from different documents $\Rightarrow$ NSP class $0$). So let's handle this next.

### Generate Sentence Pairs

Generating the sentence pairs requires some careful consideration, so we do it step by step. Firstly, for convenience and re-use, we define helper variables that hold the special tokens as defined by the tokenizer. A BERT model needs a maximum input length for both architectural and practical reasons.

In [19]:
CLS, SEP  = tokenizer.cls_token_id, tokenizer.sep_token_id
PAD, MASK = tokenizer.pad_token_id, tokenizer.mask_token_id

VOCAB_SIZE = tokenizer.vocab_size

Next, we have to consider the case where a sentence pair is too large with respect to their combined number of tokens (incl. the special tokens!). Architecturally, the learned position embedding table has a fixed number of rows (e.g., 512 in the original BERT), so there is no embedding for positions beyond that limit. Practically, self-attention compares every token with every other token, so its memory and compute cost grow quadratically with sequence length, and a limit keeps training feasible on available hardware.

A maximum length also makes batching simple: sequences are padded (or truncated) to a common length so they can be stacked into tensors of shape `(batch_size, seq_len)`. Longer texts must be truncated or split into chunks, and the pair `[CLS] A [SEP] B [SEP]` has to fit within the limit, which is why segments A and B are truncated together. Smaller limits (e.g. 128 as used in this notebook) are often used for quick experiments or early pre-training phases, since they are much cheaper, while 512 is the maximum for standard BERT.

To this end, the method `truncate_pair()` below takes in two sentences `a` and `b` and truncates them as long as the combined length is larger than the number of allowed tokens `max_tokens`. The method always removes a token from the longer sentence. However, it does randomly delete a token from the start or the end of the sentence. Trimming randomly from both ends spreads the loss evenly over the start and the end of the segment. Across many samples (and across epochs, if pairs are regenerated), different parts of the same long segment are kept, which adds diversity and acts as a mild form of data augmentation. This is the same strategy used in the original BERT preprocessing code. Of course, removing tokens from the start can cut a sentence mid-way. This is generally acceptable for MLM and NSP pre-training.

In [20]:
def truncate_pair(a, b, max_tokens, rng):
    while len(a) + len(b) > max_tokens:
        t = a if len(a) > len(b) else b          # trim the longer one
        if rng.random() < 0.5: 
            del t[0]
        else: 
            t.pop()

Finally, the method `create_instances()` generates pairs of sentences `[CLS] A [SEP] B [SEP]` where negative pairs (NSP class $1$) are formed by randomly sampling `B` from a document not containing `A`. Each pair is represented as a sequence of token IDs including the special tokens for `[CLS]` and `[SEP]`. Apart from the list of token IDs the method also returns the list of corresponding segment IDs, as well as the NSP label. To help understanding the method, here are some additional descriptions:

- **Setup:** Create a seeded random generator for reproducibility and compute `max_tokens = max_len - 3`, reserving room for `[CLS]`, `[SEP]`, `[SEP]`. Instances are collected in a list.
- **Choose a target length per document:** Normally the target is `max_tokens`. With probability `short_seq_prob` (10%) a shorter random target is used, so the model also sees short inputs, which helps at fine-tuning time.
- **Collect a chunk of consecutive sentences:** Walk through the document sentence by sentence, appending to `chunk` until its token count reaches the target or the document ends.
- **Split the chunk into A and B:** Pick a random split point `a_end`, so the first `a_end` sentences form segment A. Using a random split gives A and B varying lengths.
- **Decide the NSP label:** If the chunk has only one sentence, or with 50% probability, build a negative pair (`is_random = 1`, `NotNext`). Otherwise build a positive pair (`is_random = 0`, `IsNext`).
- **Build B for a negative pair:** Pick a random *different* document (up to 10 attempts), start at a random sentence, and append consecutive sentences until B reaches the remaining length budget (`target - len(a)`). Then rewind the index `i` so the sentences of the chunk that were not used in A are reused in the next chunk rather than wasted.
- **Build B for a positive pair:** Use the remaining sentences of the chunk, i.e. the text that truly follows A.
- **Truncate the pair:** Call `truncate_pair` to trim the longer segment until `len(a) + len(b) <= max_tokens`.
- **Assemble the instance:** Create `[CLS] A [SEP] B [SEP]` as the token ID list, and the matching segment ID list (0 for `[CLS]`, A and the first `[SEP]`; 1 for B and the last `[SEP]`). Store `(ids, types, is_random)` in `instances`.
- **Reset and continue:** Clear the chunk and move on to the next sentence until all documents are processed, then return all instances.

In [21]:
def create_instances(docs, max_len=128, short_seq_prob=0.1, seed=None):
    rng = np.random.default_rng(seed=seed)
    max_tokens = max_len - 3                     # [CLS] A [SEP] B [SEP]
    instances = []
    for d_idx, doc in enumerate(docs):
        target = max_tokens
        if rng.random() < short_seq_prob:         # a few short sequences help with
            target = rng.integers(2, max_tokens)  # short inputs at fine-tuning time
        chunk, chunk_len, i = [], 0, 0
        while i < len(doc):
            chunk.append(doc[i]); chunk_len += len(doc[i])
            if i == len(doc) - 1 or chunk_len >= target:
                a_end = 1 if len(chunk) < 2 else rng.integers(1, len(chunk))
                a = [t for s in chunk[:a_end] for t in s]

                if len(chunk) == 1 or rng.random() < 0.5:
                    is_random = 1                # NotNext
                    b_target = target - len(a)
                    r = d_idx
                    for _ in range(10):          # pick a *different* document
                        r = rng.integers(0, len(docs))
                        if r != d_idx: break
                    rdoc = docs[r]
                    start = rng.integers(0, len(rdoc))
                    b = []
                    for j in range(start, len(rdoc)):
                        b.extend(rdoc[j])
                        if len(b) >= b_target: break
                    i -= len(chunk) - a_end      # reuse sentences we didn't consume
                else:
                    is_random = 0                # IsNext
                    b = [t for s in chunk[a_end:] for t in s]

                truncate_pair(a, b, max_tokens, rng)
                ids = [CLS] + a + [SEP] + b + [SEP]              # Generate list of token IDs
                types = [0] * (len(a) + 2) + [1] * (len(b) + 1)  # Generate list of segment IDs
                instances.append((ids, types, is_random))
                chunk, chunk_len = [], 0
            i += 1
    return instances

**Important:** Note that the MLM masking and the padding is not done here. They are applied in a later step, which also lets the masking be regenerated dynamically.

We can now use `create_instances()` to create the training samples for our training, validation, and test documents.

In [22]:
train_samples = create_instances(train_docs)
val_samples   = create_instances(val_docs)
test_samples  = create_instances(test_docs)

Just to give a sense for the output, we can check the first sentence pair and its respective token IDs, segment IDs, and the NSP label.

In [23]:
tok_ids   = train_samples[0][0]
seg_ids   = train_samples[0][1]
nsp_label = train_samples[0][2]

#Convert ids from int32 to simple integers (purely for easier presentation)
tok_ids = [ int(tok_id) for tok_id in tok_ids ]

print(f"Token IDs:\n{tok_ids}\n")
print(f"Segment IDs:\n{seg_ids}\n")
print(f"NSP Label:\n{nsp_label}\n")

Token IDs:
[101, 4360, 1017, 1024, 4895, 2890, 27108, 5732, 11906, 1006, 2887, 1024, 1856, 1806, 1671, 30222, 30218, 30259, 30227, 30255, 30258, 30219, 2509, 1010, 5507, 1012, 11748, 4801, 4360, 1997, 1996, 11686, 1017, 1007, 1010, 4141, 3615, 2000, 2004, 11748, 4801, 4360, 11906, 3523, 2648, 2900, 1010, 2003, 1037, 8608, 2535, 1011, 2652, 2678, 2208, 2764, 2011, 16562, 1998, 2865, 1012, 4432, 2005, 1996, 102, 1996, 2981, 2626, 1010, 1000, 2005, 1996, 2034, 2051, 2144, 1996, 2707, 1997, 1996, 2162, 1999, 2826, 1010, 2111, 2062, 17730, 2000, 3773, 2169, 2060, 2083, 1996, 15925, 1997, 1037, 5883, 2020, 9530, 6299, 4726, 2006, 1996, 3007, 2000, 4952, 2000, 2189, 2362, 1012, 2009, 2001, 1037, 14764, 1997, 3653, 9028, 18354, 1010, 2188, 2000, 2070, 1997, 2214, 8936, 1005, 1055, 2190, 102]

Segment IDs:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

### Generate Datesets

Since we are using PyTorch, it is strongly recommended and convenient to utilize the provided `Dataset` and `DataLoader` because they turn data handling into a standard, reusable interface that fits PyTorch's training loop. A `Dataset` defines what a single sample looks like (via `__len__` and `__getitem__`), while the `DataLoader` takes care of everything needed to feed the model efficiently: shuffling samples each epoch, grouping them into batches, padding them to a common length through a custom `collate_fn`, and loading data in parallel worker processes so the GPU is not left waiting. This keeps the training code short and clean, avoids hand-written batching logic, makes it easy to apply on-the-fly steps such as dynamic MLM masking, and allows the same data pipeline to be reused for training, validation, and other models.

So let's start with the `Dataset` implementation. The `BertPretrainDataset` class wraps the instances produced by `create_instances` in the standard PyTorch `Dataset` interface, so they can be served by a `DataLoader`. On creation, it generates the instances from the documents, `__len__` returns how many there are, and `__getitem__` returns one instance as a tuple of token IDs, segment IDs, and the NSP label. The `regenerate()` method rebuilds all instances with a new random seed, which draws new sentence pairs, new random negatives, and new splits. Calling it between epochs adds diversity, in line with the earlier discussion of recreating pairs.

In [24]:
class BertPretrainDataset(Dataset):
    def __init__(self, docs, max_len=128):
        self.docs, self.max_len = docs, max_len
        self.regenerate()

    def regenerate(self, seed=None):
        self.instances = create_instances(self.docs, self.max_len, seed=seed)
    
    def __len__(self): 
        return len(self.instances)
    
    def __getitem__(self, i): 
        return self.instances[i]

Now its finally time to handle the masking got the MLM learning objective. For this, the `mask_tokens` method applies BERT's MLM corruption to a batch of token IDs. It first selects about 15% of the positions at random, never choosing the special tokens `[PAD]`, `[CLS]`, and `[SEP]`, and builds a label tensor that keeps the original token ID at the selected positions and `-100` everywhere else, so the loss is computed only where a prediction is required. Among the selected positions, 80% are replaced by `[MASK]`, 10% by a random vocabulary token, and the remaining 10% are left unchanged. The function returns the corrupted input IDs together with the labels.

In [25]:
def mask_tokens(input_ids, mlm_prob=0.15):
    labels = input_ids.clone()
    special = (input_ids == PAD) | (input_ids == CLS) | (input_ids == SEP)

    prob = torch.full(input_ids.shape, mlm_prob).masked_fill(special, 0.0)
    masked = torch.bernoulli(prob).bool()
    labels[~masked] = -100                       # loss only on selected positions

    to_mask = torch.bernoulli(torch.full(input_ids.shape, 0.8)).bool() & masked
    input_ids[to_mask] = MASK                    # 80% -> [MASK]

    to_rand = torch.bernoulli(torch.full(input_ids.shape, 0.5)).bool() & masked & ~to_mask
    input_ids[to_rand] = torch.randint(VOCAB_SIZE, input_ids.shape)[to_rand]   # 10% -> random
    return input_ids, labels                     # remaining 10% stay unchanged

Since masking is cheap and random, we perform **dynamic masking** where every time a training sample is served it receives a fresh masking pattern. The method `collate()` below implements both dynamic masking as well as dynamic padding. It right-pads all sequences to the longest one in the batch (`[PAD]` for the IDs, $0$ for the segment IDs), builds the attention mask ($1$ for real tokens, $0$ for padding), and stacks the NSP labels. It then calls `mask_tokens` on the padded input IDs to apply the MLM corruption and create the MLM labels.The function returns a dictionary containing the corrupted `input_ids`, `token_type_ids`, `attention_mask`, `mlm_labels`, and `nsp_labels`.

In [26]:
def collate(batch):
    B, L = len(batch), max(len(x[0]) for x in batch)
    tok_ids = torch.full((B, L), PAD, dtype=torch.long)
    seg_ids = torch.zeros((B, L), dtype=torch.long)
    attention_mask = torch.zeros((B, L), dtype=torch.long)
    for k, (ids, types, _) in enumerate(batch):
        n = len(ids)
        tok_ids[k, :n] = torch.tensor(ids)
        seg_ids[k, :n] = torch.tensor(types)
        attention_mask[k, :n] = 1
    nsp_labels = torch.tensor([x[2] for x in batch])
    tok_ids, mlm_labels = mask_tokens(tok_ids)
    return dict(tok_ids=tok_ids, seg_ids=seg_ids, attention_mask=attention_mask, mlm_labels=mlm_labels, nsp_labels=nsp_labels)

In the last step, we can now create the `Dataset` and `DataLoader` instances. Since this notebook is for educational purposes we only use a maximum length of $128$ compared to the $512$ of the original BERT model. Notice that each `DataLoader` is given the `collate()` method. When iterating over all batches, each data loader will apply the `collate()` method to the batch &mdash; thus performing MLM masking and padding &mdash; before passing the batch as input to the model.

In [27]:
train_ds = BertPretrainDataset(train_docs, max_len=128)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate)

val_ds = BertPretrainDataset(val_docs, max_len=128)
val_dl = DataLoader(val_ds, batch_size=64, collate_fn=collate)

test_ds = BertPretrainDataset(test_docs, max_len=128)
test_dl = DataLoader(test_ds, batch_size=64, collate_fn=collate)

**Important:** The datasets used in this notebook (`WikiText-2` and, at most, `WikiText-103`) are small enough that the tokenized documents, the generated NSP instances, and the resulting batches all fit comfortably into RAM. This keeps the code simple: lists of token ID arrays can be created once, accessed instantly by index, and regenerated between epochs without any special infrastructure, which lets us focus on the concepts rather than on data engineering.

In a real-world setting, pre-training corpora are many orders of magnitude larger (the original BERT used about 3.3B words, and modern models use trillions of tokens), so they cannot be held in memory, and the data pipeline becomes a serious engineering concern. Typical considerations include storing the corpus in sharded files on disk or in cloud storage, tokenizing it once offline and saving compact binary formats (e.g. memory-mapped arrays or Arrow files), streaming samples lazily instead of loading everything, and using multiple `DataLoader` workers to prefetch data so the GPUs are never left waiting. It also requires distributed sampling across many devices, deduplication and quality filtering of the text, and reproducible shuffling and resumption of training after interruptions. Because these concerns can dominate the effort of large-scale pre-training, it is worth keeping in mind that our in-memory approach is a deliberate simplification.

---

## Model Training

With the train data in place and ready to go, we can now train a basic BERT model from scratch. We consider two approaches because they serve complementary goals: understanding and practicality.

* **PyTorch-only.** Here the model is assembled from PyTorch's basic building blocks, namely `nn.Embedding` for the token, segment, and position tables, `TransformerEncoderLayer` and `TransformerEncoder` for the stack of encoder layers, and small linear layers for the MLM and NSP heads. Writing these pieces ourselves makes the architecture explicit: we see how the three embeddings are summed, how the attention mask is passed to the encoder, how the `[CLS]` vector feeds the NSP head, and how the two losses are combined. This is the better route for understanding what BERT actually consists of, and it shows that the model is not a black box but a fairly compact composition of familiar components.

* **`transformers` library.** Here we use wrapper classes such as `BertConfig` to define the architecture (number of layers, hidden size, vocabulary size, etc.) and `BertForPreTraining` to obtain a complete BERT with both pre-training heads. This is the more practical approach: the code is much shorter, the implementation is mature, tested, and optimized, and the resulting model is directly compatible with the rest of the ecosystem (tokenizers, the `Trainer`, saving and sharing models, and fine-tuning classes). It is also what one would typically use in real projects.

Going through both is useful because the PyTorch-only version gives the conceptual insight, while the `transformers` version shows how the same model is built efficiently in practice. Comparing them also confirms that the library classes implement exactly the components we built by hand.

### Approach 1: PyTorch-Only

The PyTorch-only approach is naturally more involved because nothing is given for free: we have to create the embedding matrices, wire up the encoder, attach the two pre-training heads, and handle details such as the attention mask format and weight initialization ourselves. This extra effort is the point. Having to write each piece forces us to see exactly what goes into BERT and how data flows through it, which a single call like `BertForPreTraining(config)` hides.

To keep the implementation organized, we split it into two parts that mirror the architecture. A custom `BertEmbeddings` class implements the input stage: it looks up the token, segment, and position embeddings, sums them, and applies layer normalization and dropout, producing the `(batch, seq_len, d_model)` input for the encoder. The actual `BertModel` then uses this embedding class, passes its output through a stack of encoder layers (`TransformerEncoderLayer` and `TransformerEncoder`) with the padding mask, and adds the MLM head on all positions and the NSP head on the `[CLS]` vector.

#### Embeddings

BERT represents each input position in a training sample by three IDs, each looked up in its own embedding matrix. The three resulting vectors are summed to form the input to the first encoder layer. These three types of IDs are:

* **Token ID:** the index of the WordPiece token in the vocabulary. It tells the model *which* token is at this position and selects the token embedding, which carries lexical meaning. Which ID refers to which token depends on the pretrained tokenizer. For example, using the `bert-base-uncased` tokenizer, the IDs for the special tokens are: `[CLS]` = $101$, `[SEP]` = $102$, `[MASK]` = $103$, `[PAD]` = $0$. Examples for "normal" tokens are `movie` = $3185$, `all` = $2035$, or `banana` = $15212$.
* **Segment ID:** indicates which segment the token belongs to:
    * $0$ for `[CLS]`, segment A and the first `[SEP]`
    * $1$ for segment B and the final `[SEP]`
* **Position ID:** the index of the token in the sequence ($0$, $1$, $2$, ...). It selects the position embedding, which is needed because self-attention has no built-in notion of word order.

The table below shows how each token in our example input string gets converted into these three types of IDs. The token IDs are the true values when using the pretrained `bert-base-uncased` tokenizer.

| Index | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 | 14 | 15 | 16 | 17 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| **Token** | `[CLS]` | I | `[MASK]` | a | movie | last | <font color="red"><b>banana</b></font> | `[SEP]` | The | special | `[MASK]` | were | all | <font color="red"><b>so</b></font> | cool | `[SEP]` | `[PAD]` | `[PAD]` |
| **Token ID** | 101 | 1045 | 103 | 1037 | 3185 | 2197 | 15212 | 102 | 1996 | 2569 | 103 | 2020 | 2035 | 2061 | 4658 | 102 | 0 | 0 |
| **Segment ID** | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 0 | 0 |
| **Position ID** | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 | 14 | 15 | 16 | 17 |

Two things are worth mentioning here. Firstly, the segment ID for the final `[PAD]` tokens can be $0$ or $1$ since we ignore those tokens using the attention mask as well as the loss mask. And secondly, this example uses very common words, meaning that no word is needed to be split into subwords. This keeps the example easier to read. However, as discussed earlier, having subwords does not change the overall process.

However, these IDs cannot be fed directly into the Transformer encoder which expects a single embedding vector for each token, since the encoder's matrix multiplications, attention scores, and non-linearities need continuous, fixed-size vectors to operate on. We there need to convert token IDs, segment IDs, and position IDs into their respective embeddings and then combine to get the final input embeddings we can pass to the encoder.

* **Token embeddings.** The integer labels representing the token IDs capture no semantics. For example, the number $3185$ for `movie` is not "larger" or "closer to" $3186$ (`shook`) in any meaningful sense. Feeding raw integers would impose a false numerical ordering and give the network no way to represent similarity between tokens. Each ID is therefore used as an index into a learnable embedding matrix $\mathbf{E}_{\text{token}}$, a matrix of shape `(vocab_size, d_model)`, which returns a dense vector of dimension `d_model` (e.g. $768$; standard size for a basic BERT model). Because $\mathbf{E}_{\text{token}}$ is learned during training, the embeddings are optimized for the learning objectives: tokens used in similar contexts receive similar vectors, which is exactly the Distributional Hypothesis at work. This is why the IDs are only lookup keys, and the embeddings are the actual input to the model.

* **Segment embeddings.** Just like the token IDs, the values $0$ and $1$ for the two segments are categorical labels, not quantities. Segment B is not "one more" than segment A, so feeding the raw integer would suggest a numerical relationship that does not exist. Of course, this means that the learnable segment embedding matrix $\mathbf{E}_{\text{segment}}$ has a shape of `(2, d_model)`. Every token in segment A receives the same vector, and every token in segment B receives another. 

* **Position embeddings.** Using the same reasoning, say, position 5 is not "five times" position 1 in any semantic sense, and a raw integer would give the network no useful way to compare or combine positions. Although, in principle, inputs can be of arbitrary lengths, defining the learnable embedding matrix $\mathbf{E}_{\text{token}}$ requires to specify the number of rows, i.e., the maximum length `max_len` of an input. Thus $\mathbf{E}_{\text{position}}$ has the shape `(max_len, d_model)`, with $512$ the maximum length of the original BERT model. Adding it to the token embedding injects word-order information, which self-attention lacks since it treats the input as an unordered set.

With these three embedding matrices, for a position $i$ and the corresponding token $t_i$ and the segment $s_i$, the embedding vector $\mathbf{e}_i$ is computed by adding all three types of embeddings element wise. This works since all three embedding matrices produce vectors of the same size `d_model`:

$$\large
\mathbf{e}_i = \mathbf{E}_{\text{token}}[t_i] + \mathbf{E}_{\text{segment}}[s_i] + \mathbf{E}_{\text{position}}[i]
$$

After summing the embeddings, BERT applies layer normalization and dropout before the first encoder layer. The three embedding matrices are initialized and learned independently, so their sum can have an uneven scale across positions and features. Layer normalization standardizes each position's vector across the feature dimension (with learned scale and shift), giving the encoder a stable, well-scaled input and helping training converge, which matters especially because the first sublayer's residual connection carries this input through the whole stack. Dropout randomly zeroes some of the values during training, acting as regularization: the embedding matrices hold a large share of the model parameters, so they are prone to overfitting, and dropout discourages the model from relying on any single embedding dimension.

The class `BertEmbeddings` implements all these components and steps; the three instances of `nn.Embeddings` and their respective inputs sizes reflecting the mapping of the token IDs, segment IDs, and position IDs.

In [28]:
class BertEmbeddings(nn.Module):
    def __init__(self, vocab_size=30522, hidden_size=768, max_len=512, n_segments=2, pad_token_id=0, layer_norm_eps=1e-12, dropout=0.1):
        super().__init__()
        self.tok_embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=pad_token_id)
        self.seg_embeddings = nn.Embedding(n_segments, hidden_size)
        self.pos_embeddings = nn.Embedding(max_len, hidden_size)

        self.LayerNorm = nn.LayerNorm(hidden_size, eps=layer_norm_eps)
        self.dropout = nn.Dropout(dropout)

        # Fixed position ids [0, 1, ..., max_len-1], shape (1, max_len)
        self.register_buffer("position_ids", torch.arange(max_len).unsqueeze(0), persistent=False)

    def forward(self, tok_ids, seg_ids=None):
        batch_size, seq_len = tok_ids.shape

        if seg_ids is None: seg_ids = torch.zeros_like(tok_ids)
        pos_ids = self.position_ids[:, :seq_len]            

        tok_emb = self.tok_embeddings(tok_ids)     # (B, T, H)
        seg_emb = self.seg_embeddings(seg_ids)     # (B, T, H)
        pos_emb = self.pos_embeddings(pos_ids)     # (1, T, H), broadcasts

        embeddings = tok_emb + seg_emb + pos_emb
        embeddings = self.LayerNorm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings

Appreciate that the position IDs do not depend on the content of the input, only on its length: for any sequence they are simply 0, 1, 2, ..., `seq_len - 1`. Unlike token IDs (which come from the tokenizer) and segment IDs (which depend on where segment B starts), they carry no information that must be supplied from outside. The `BertEmbeddings` class can therefore create them itself, once, in its constructor using `self.register_buffer()`. This stores a fixed tensor `[[0, 1, ..., max_len - 1]]` of shape `(1, max_len)`. In the `forward()` method, the embeddings class only needs to slice off the first `seq_len` entries, `self.position_ids[:, :seq_len]`, and look them up in the position embedding matrix. 

Registering the position IDs as a buffer ties them to the module: they move automatically with `.to(device)`, so they are never on the wrong device and need not be recreated in every forward pass. They are not trained, and with `persistent=False` they are left out of saved checkpoints, since the constructor can recreate them deterministically. This keeps the interface simple, because the caller only passes `tok_ids` and `seg_ids`, and it prevents inconsistent positions. Position *i* simply selects row *i* of the position embedding table, and padding does not change this because `[PAD]` is handled by the attention mask. The only limitation is that `seq_len` must not exceed `max_len`, since longer sequences have no embedding row.

#### Output Class

The `BertPreTrainingOutput` class below has no effect on how our custom model learns. Its only purpose is to make the output of `MyBertModel` look like the output of `BertForPreTraining` in the `transformers` library, which returns an object with the fields `prediction_logits` (MLM) and `seq_relationship_logits` (NSP) rather than a plain tuple.

In [29]:
@dataclass
class BertPreTrainingOutput:
    prediction_logits: torch.Tensor        # MLM logits: (batch, seq_len, vocab_size)
    seq_relationship_logits: torch.Tensor  # NSP logits: (batch, 2)

Matching this interface has a practical benefit: the same code for computing the loss, evaluating, and training can be used for both models without changes, so we only have to swap the model object. The class is deliberately minimal. It contains no logic, just a container with two tensors. The `transformers` output class has additional optional fields (such as `loss`, `hidden_states`, and `attentions`), which we do not need here.

#### Model Class

The custom `MyBertModel` class in the code cell below assembles the complete pre-training model. PyTorch's `TransformerEncoderLayer` and `TransformerEncoder` already provide the generic core (multi-head self-attention, feedforward network, residual connections, layer normalization, dropout), so we only need to add what is specific to BERT around them:

- **Embeddings (`BertEmbeddings`):** turns the three ID types (token, segment, position) into one summed, normalized, dropout-regularized `(batch, seq_len, d_model)` input. The encoder only accepts continuous vectors, so this is the bridge from integer IDs to the encoder.
- **Encoder stack (`TransformerEncoderLayer` + `TransformerEncoder`):** the stack of layers that computes the contextual embeddings. It receives the padding information so that `[PAD]` positions are ignored in attention.
- **Pooler:** a linear layer plus tanh applied to the `[CLS]` vector. It produces a single sentence-pair-level representation, which serves as the input to the NSP head (and is often reused for classification during fine-tuning).
- **NSP head:** a linear layer mapping the pooled vector to two logits (`IsNext`/`NotNext`). It provides the sentence-level training signal.
- **MLM head:** a small transformation (linear layer, GELU, layer normalization) followed by a projection to vocabulary size, applied to every position. It produces the logits for predicting the original tokens at the masked positions.
- **Weight tying:** the output projection of the MLM head shares its weight matrix with the token embedding table. Both map between the vocabulary and the `d_model` space, so sharing them removes parameters, and it ties the input and output representations of a word together, which acts as a helpful regularizer and often improves quality.

Together, these components cover everything between raw IDs and the two losses. The embeddings feed the encoder, the encoder produces contextual embeddings, and the two heads turn those embeddings into the MLM and NSP predictions that are trained with cross-entropy.

In [30]:
class MyBertModel(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_layers=2, n_heads=4, d_ff=512, max_len=128, dropout=0.1, pad_token_id=0):
        super().__init__()
        
        self.embeddings = BertEmbeddings(vocab_size, d_model, max_len, dropout=dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",   # BERT uses GELU instead of ReLU
            batch_first=True,    # tensors are (batch, seq, features)
            norm_first=False,    # post-LayerNorm, as in the original BERT
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers, enable_nested_tensor=False)

        # Pooler: transform the [CLS] vector (used for NSP and classification)
        self.pooler = nn.Sequential(
            nn.Linear(d_model, d_model), 
            nn.Tanh()
        )

        # NSP head
        self.nsp_head = nn.Linear(d_model, 2)

        # MLM head: dense + GELU + LayerNorm, then project to the vocabulary
        self.mlm_transform = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model)
        )
        self.mlm_decoder = nn.Linear(d_model, vocab_size, bias=False)
        self.mlm_decoder.weight = self.embeddings.tok_embeddings.weight  # weight tying
        self.mlm_bias = nn.Parameter(torch.zeros(vocab_size))

        self.apply(self._init_weights)
        self.embeddings.tok_embeddings.weight.data[pad_token_id].zero_()   # pad_token_id = 0 by default

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)
        elif isinstance(module, nn.MultiheadAttention):
            nn.init.normal_(module.in_proj_weight, mean=0.0, std=0.02)   # fused Q/K/V
            if module.in_proj_bias is not None:
                nn.init.zeros_(module.in_proj_bias)

    def forward(self, tok_ids, seg_ids, attention_mask):

        x = self.embeddings(tok_ids, seg_ids)

        # PyTorch expects True at positions that should be IGNORED (padding)
        padding_mask = attention_mask == 0
        hidden = self.encoder(x, src_key_padding_mask=padding_mask)  # (B, T, d_model)

        mlm_logits = self.mlm_decoder(self.mlm_transform(hidden)) + self.mlm_bias
        nsp_logits = self.nsp_head(self.pooler(hidden[:, 0]))         # [CLS] is position 0

        return BertPreTrainingOutput(
            prediction_logits=mlm_logits,
            seq_relationship_logits=nsp_logits,
        )

The `forward()` method of the model class only computes predictions: it returns the **MLM logits** of shape `(batch, seq_len, vocab_size)`, one score per vocabulary token at every position, and the **NSP logits** of shape `(batch, 2)`, one score for `IsNext` and `NotNext` per sentence pair. Keeping the loss out of `forward()` separates the model (what it predicts) from the training objective (how wrong it is), which makes the model reusable for inference, feature extraction, and evaluation, and makes it easy to change or log the losses separately. The `compute_loss()` method turns these logits into the two training losses using cross-entropy.

- **MLM loss:** The logits are flattened to `(batch * seq_len, vocab_size)` and the labels to `(batch * seq_len)`, since `F.cross_entropy` expects one row of class scores per prediction. `ignore_index=-100` implements the loss masking: all positions with label `-100` (non-selected tokens, `[CLS]`, `[SEP]`, `[PAD]`) are skipped, so the loss is averaged only over the ~15% selected positions.
- **NSP loss:** The two NSP logits per pair are compared directly with the binary NSP label, one prediction per sentence pair.
- **Numerical details:** `F.cross_entropy` applies the softmax internally (in a numerically stable way), so it takes raw logits rather than probabilities. The `.float()` casts make sure the loss is computed in 32-bit precision, which matters when the model runs in mixed precision (e.g. float16 or bfloat16).

The two losses are returned separately so they can be logged individually, and the total pre-training loss is typically their sum, `mlm_loss + nsp_loss`, which is then backpropagated through the whole model.

In [31]:
def compute_loss(mlm_logits, nsp_logits, batch):
    mlm_loss = F.cross_entropy(mlm_logits.view(-1, mlm_logits.size(-1)).float(), batch["mlm_labels"].view(-1), ignore_index=-100)
    nsp_loss = F.cross_entropy(nsp_logits.float(), batch["nsp_labels"])
    return mlm_loss, nsp_loss

We now have all the ingredients to instantiate a `MyBertModel`. Here we deliberately choose a small configuration by default (hidden size $128$, $2$ encoder layers, $4$ attention heads, feedforward size $512$, maximum length $128$) instead of the orginal BERT model. WikiText is tiny compared with the original pre-training corpus, and a model this size trains in minutes on a laptop or a single GPU or even CPU, so we can run the complete pipeline and observe the loss decreasing without large compute. The `max_len` of 128 matches the maximum length used when creating the instances, so every position has a row in the position embedding table.

Calling `.to(DEVICE)` moves all parameters and buffers (including the position IDs) to the same device as the batches, avoiding device mismatch errors. Printing the parameter count is a useful sanity check on the model's scale. Because the vocabulary of about 30k tokens makes the token embedding table (and the tied MLM output layer) a large share of the weights, the total is dominated by the embeddings rather than by the two encoder layers, which is worth noticing when comparing with the roughly 110M parameters of orIginal basic BERT model.

In [32]:
model = MyBertModel(vocab_size=VOCAB_SIZE, d_model=128, n_layers=2, n_heads=4, d_ff=512, max_len=128).to(DEVICE)

print(f"Model size: {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

Model size: 4.4M parameters


#### Training

When training any model, we typically keep track of the progress by looking at the trend of the training loss. However, the training loss alone can be misleading. It measures how well the model fits the data it is optimized on, so it keeps decreasing even when the model starts to memorize specific sentence pairs and masking patterns instead of learning general language structure. Only a held-out validation set that the model never trains on shows whether the learned representations generalize. Tracking both curves helps diagnose problems: if both fall together, learning works; if training loss falls while validation loss stalls or rises, the model is overfitting; if both stay high, the model is underfitting or the setup has a bug. It also guides decisions such as when to stop, which checkpoint to keep, and how to set the learning rate.

The `evaluate()` method computes these validation metrics in a compact way:

- **No training behavior:** `@torch.no_grad()` disables gradient tracking to save memory and time, and `model.eval()` switches off dropout so the predictions are deterministic. `model.train()` restores training mode at the end.
- **Limited cost:** `max_batches=200` evaluates only a subset, so validation stays fast even when run frequently during training.
- **Fixed randomness:** The validation batches are masked on the fly in the collate function, so the corruption is random. `torch.random.fork_rng()` with `torch.manual_seed(0)` makes the masking identical at every evaluation, which makes validation losses comparable across epochs, and the global random state of training is restored afterwards.
- **Metrics:** It averages the MLM and NSP losses over the batches, and converts the MLM loss into **perplexity** via `exp(loss)`, which is easier to interpret (roughly the number of equally likely candidates the model is choosing among at a masked position). It also computes the **NSP accuracy**, the fraction of sentence pairs correctly classified as `IsNext` or `NotNext`.

Together, these numbers show whether the model is learning and generalizing, not just fitting the training data.

In [33]:
@torch.no_grad()
def evaluate(model, dl, max_batches=200):
    model.eval()
    mlm_sum = nsp_sum = 0.0
    correct = n_batches = n_samples = 0
    with torch.random.fork_rng():
        torch.manual_seed(0)
        for i, batch in enumerate(dl):
            if i >= max_batches: break
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(batch["tok_ids"], batch["seg_ids"], batch["attention_mask"])
            mlm_loss, nsp_loss = compute_loss(out.prediction_logits, out.seq_relationship_logits, batch)

            mlm_sum += mlm_loss.item()
            nsp_sum += nsp_loss.item()
            correct += (out.seq_relationship_logits.argmax(-1) == batch["nsp_labels"]).sum().item()
            n_batches += 1
            n_samples += batch["nsp_labels"].size(0)
    model.train()

    mlm_loss = mlm_sum / n_batches
    return dict(
        mlm_loss=mlm_loss,
        ppl=float(torch.exp(torch.tensor(mlm_loss))),
        nsp_loss=nsp_sum / n_batches,
        nsp_acc=correct / n_samples,
    )

The last requirement for training the model is to set up the optimizer that updates the learnable weights based on the gradients computed during backpropagation. In this notebook, we closely follow the setup used for training the original BERT model by implementing the following two auxiliary methods:

* `make_optimizer()` creates the AdamW optimizer, which is Adam with *decoupled* weight decay: the decay is applied directly to the weights rather than being mixed into the gradient, which works better with adaptive optimizers and acts as regularization against overfitting. The parameters are split into two groups. Weights (matrices, `ndim >= 2`) receive weight decay of 0.01, while biases and layer normalization parameters (1D tensors, or names containing "norm") receive none. Decaying these small parameters would only pull scales and offsets toward zero without a regularizing benefit and can hurt training, so excluding them is standard practice.

* `linear_warmup_decay()` returns a scheduler that scales the base learning rate by a factor over the course of training. For the first `warmup_steps` the factor rises linearly from 0 to 1, and afterwards it decays linearly down to 0 at `total_steps`. The warmup matters because early in training the weights are random, the gradients are large and noisy, and Adam's moment estimates are still unreliable, so a full learning rate right away can destabilize training. This is especially true for Transformers with layer normalization. The linear decay then lets the model take large steps early and progressively finer steps toward the end, which helps it settle into a good solution.
the much shorter training run, but the shape of the schedule is the same.

Together, the optimizer defines *how* the weights are updated, and the scheduler defines *how big* those updates are at each step. 

In [34]:
def make_optimizer(model, lr=5e-4, weight_decay=0.01):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        (no_decay if p.ndim < 2 or "LayerNorm" in name or "norm" in name else decay).append(p)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": weight_decay},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr, betas=(0.9, 0.999), eps=1e-6)

def linear_warmup_decay(optimizer, warmup_steps, total_steps):
    def f(step):
        if step < warmup_steps: return step / max(1, warmup_steps)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, f)

To create an instance of the optimizer, we first need to decide on the number of epochs, i.e., the number of passes over the training data. Here we set `EPOCHS = 5` by default to ensure that training will not take too long &mdash; again, the goal is not to learn state-of-the-art embeddings but to clearly explain all involved steps. `LOG_EVERY = 100` and `EVAL_EVERY = 500` control how often (in steps) the training loss is logged and the validation metrics from `evaluate()` are computed. 

With the number of epochs, we can compute `total_steps`, i.e, the total number of optimizer updates, which the learning rate schedule needs to know in order to decay to zero exactly at the end of training. The optimizer is created with `make_optimizer`, which gives AdamW with selective weight decay, and the scheduler wraps it with a linear warmup over the first $6\%$ of the steps  followed by a linear decay to $0$. The $6\%$ warmup is a common rule of thumb that scales the warmup to the length of the run, whereas the original BERT used $10k$ warmup steps (about $1\%$ of its $1M$ training steps). Note that the scheduler must later be advanced with `scheduler.step()` once per optimizer step inside the training loop, not once per epoch.

In [35]:
EPOCHS, LOG_EVERY, EVAL_EVERY = 5, 100, 500

total_steps = EPOCHS * len(train_dl)
optimizer = make_optimizer(model)
scheduler = linear_warmup_decay(optimizer, int(0.06 * total_steps), total_steps)

Finally, we can train the model using the standard PyTorch training loop for the custom `MyBertModel` model, and the core of it is only a few lines. The actual training (i.e., the essential part) iterates over all training batches, and for each batch runs the forward pass to get the MLM and NSP logits, computes the two losses with `compute_loss()`, and sums them into one total loss. `loss.backward()` computes the gradients, gradient clipping (max norm 1.0) keeps updates stable, `optimizer.step()` updates the weights, `optimizer.zero_grad()` clears the gradients, and `scheduler.step()` advances the learning rate schedule. At the start of every epoch after the first, `train_ds.regenerate()` creates new sentence pairs and negatives, which adds diversity between epochs.

Most of the loop's remaining code exists to monitor training rather than to change the model. It keeps running averages of the training losses, which smooths out noisy batch-level values, and records them in a `history` dictionary so learning curves can be plotted afterwards. At regular intervals it prints the epoch, step, losses, learning rate, and elapsed time, and it periodically evaluates the model on the validation set to check that it generalizes, reporting the MLM loss, perplexity, and NSP accuracy. Finally, it saves the model weights as a checkpoint at each evaluation so training can be resumed or the model reused later. Together, these steps show whether the model is actually learning and not overfitting.

In short, the few lines of forward pass, loss, backward pass, and update do the learning, while the rest record the training and validation curves so we can verify that the model is actually learning and not overfitting. Keep in mind that the training time will strongly depend on your hardware, including the availability of a GPU. As a crude estimate, on a modern CPU, the code cell below may run for several minutes and up to half an hour.

In [36]:
history = dict(step=[], train_mlm=[], train_nsp=[], val_step=[], val_mlm=[], val_nsp_acc=[])
step, t0, run_mlm, run_nsp, run_n = 0, time.time(), 0.0, 0.0, 0

model.train()

for epoch in range(EPOCHS):
    if epoch > 0:
        train_ds.regenerate()
    for batch in train_dl:
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

        out = model(batch["tok_ids"], batch["seg_ids"], batch["attention_mask"])
        mlm_loss, nsp_loss = compute_loss(out.prediction_logits, out.seq_relationship_logits, batch)
        loss = (mlm_loss + nsp_loss)
        loss.backward()

        run_mlm += mlm_loss.item(); run_nsp += nsp_loss.item(); run_n += 1

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        step += 1

        if step % LOG_EVERY == 0:
            history["step"].append(step)
            history["train_mlm"].append(run_mlm / run_n); history["train_nsp"].append(run_nsp / run_n)
            print(f"Epoch {epoch} step {step}/{total_steps} | MLM {run_mlm/run_n:.3f} | "
                  f"NSP {run_nsp/run_n:.3f} | lr {scheduler.get_last_lr()[0]:.2e} | {time.time()-t0:.0f}s")
            run_mlm = run_nsp = 0.0; run_n = 0

        if step % EVAL_EVERY == 0 or step == total_steps:
            m = evaluate(model, val_dl)
            history["val_step"].append(step); history["val_mlm"].append(m["mlm_loss"])
            history["val_nsp_acc"].append(m["nsp_acc"])
            print(f"  [val] mlm {m['mlm_loss']:.3f} (ppl {m['ppl']:.1f}) | nsp acc {m['nsp_acc']:.3f}")
            torch.save(model.state_dict(), f"{output_folder}bert_mini.pt")

Epoch 0 step 100/1795 | MLM 9.389 | NSP 0.696 | lr 4.67e-04 | 7s
Epoch 0 step 200/1795 | MLM 7.360 | NSP 0.693 | lr 4.72e-04 | 13s
Epoch 0 step 300/1795 | MLM 7.273 | NSP 0.692 | lr 4.43e-04 | 19s
Epoch 1 step 400/1795 | MLM 7.184 | NSP 0.691 | lr 4.13e-04 | 26s
Epoch 1 step 500/1795 | MLM 7.112 | NSP 0.691 | lr 3.84e-04 | 32s
  [val] mlm 7.084 (ppl 1193.0) | nsp acc 0.515
Epoch 1 step 600/1795 | MLM 7.035 | NSP 0.691 | lr 3.54e-04 | 39s
Epoch 1 step 700/1795 | MLM 7.005 | NSP 0.692 | lr 3.24e-04 | 45s
Epoch 2 step 800/1795 | MLM 6.962 | NSP 0.690 | lr 2.95e-04 | 53s
Epoch 2 step 900/1795 | MLM 6.948 | NSP 0.689 | lr 2.65e-04 | 59s
Epoch 2 step 1000/1795 | MLM 6.903 | NSP 0.687 | lr 2.35e-04 | 65s
  [val] mlm 6.903 (ppl 995.1) | nsp acc 0.544
Epoch 2 step 1100/1795 | MLM 6.878 | NSP 0.686 | lr 2.06e-04 | 72s
Epoch 3 step 1200/1795 | MLM 6.873 | NSP 0.677 | lr 1.76e-04 | 80s
Epoch 3 step 1300/1795 | MLM 6.836 | NSP 0.664 | lr 1.47e-04 | 86s
Epoch 3 step 1400/1795 | MLM 6.835 | NSP 0.644

The training output shows the pattern we hoped for: the MLM and NSP losses decrease on both the training and the validation data, and the NSP accuracy on the validation set increases. Since the validation metrics improve together with the training metrics, the model is not merely memorizing the training pairs, and the full pipeline works end to end. The sentence pairs, the 80/10/10 masking, the loss masking, the embeddings, the encoder, the two heads, and the optimizer with its warmup and decay schedule all interact correctly, because a bug in any of them would typically show up as a loss that stays flat or diverges.

However, we should not expect the model to have learned very reliable embeddings after only a few epochs on such a small dataset with a tiny model (2 layers, hidden size 128). The MLM perplexity will still be high, and the NSP task can often be solved early through simple topic cues, so its accuracy says little about deeper language understanding. This is not the purpose here, though. The goal is to understand how BERT is trained and to verify that every component works as intended. Obtaining strong embeddings would require far more data, a larger model, and much longer training, which is why in practice we start from pretrained models.

**Important:** Before training the second model with the `transformers` library, we release the memory still held by the first one. The custom model, its optimizer (AdamW keeps two extra state tensors per parameter), and the scheduler all keep tensors alive on the device, and if they stay allocated the second model may run out of memory or train more slowly. The cleanup works in three steps:

- **`del model, optimizer, scheduler`:** removes the Python references, since memory can only be freed once nothing points to it. Leftover variables such as `batch`, `loss`, or the logits should be deleted too.
- **`gc.collect()`:** runs Python's garbage collector to reclaim objects that are no longer referenced, including reference cycles.
- **`torch.cuda.empty_cache()` / `torch.mps.empty_cache()`:** PyTorch keeps freed GPU memory in a cache for reuse, so these calls return the unused blocks to the system. The first applies to NVIDIA GPUs and the second to Apple Silicon. Each is wrapped in `try/except`, so the cell runs on any machine, including a CPU-only one, where neither call is needed.

In [37]:
try:
    del model, optimizer, scheduler
except:
    pass
gc.collect()
try:
    torch.cuda.empty_cache()
except:
    pass
try:
    torch.mps.empty_cache()
except:
    pass

If needed, the weights of the first model are safe on disk in `bert_mini.pt`.

### Approach 2: `transformers` Library

Having built and trained BERT by hand, we can now see how much of that work the `transformers` library packages into ready-made classes. Each component we implemented has a counterpart: `BertConfig` replaces the manual choice of hyperparameters (hidden size, layers, heads, vocabulary size, maximum length), and `BertForPreTraining` replaces our custom embeddings, encoder stack, pooler, MLM and NSP heads, and weight tying, including the loss computation when labels are passed. Data preparation is also covered: `DataCollatorForLanguageModeling` applies the 80/10/10 masking and builds the labels with `-100`, and the `Trainer` takes over the training loop with the optimizer, warmup and decay schedule, gradient clipping, logging, evaluation, and checkpointing.

It is like speed-running the same steps: the workflow is identical, but each step shrinks from dozens of lines to a single call, and it relies on mature, tested, and optimized implementations. This is the approach used in practice, and having built the model by hand first makes it clear what these wrappers do internally, so they are no longer a black box. To this end, in the code cell below, we first use the `BertConfig` class to define all model parameters &mdash; by default, we use the same as for the manual approach. Using this configuration, we can create an instance of `BertForPreTraining`.

In [38]:
config = BertConfig(
    vocab_size=VOCAB_SIZE,
    hidden_size=128,
    num_hidden_layers=2,
    num_attention_heads=4,
    intermediate_size=512,
    max_position_embeddings=128,
    type_vocab_size=2,
    pad_token_id=tokenizer.pad_token_id,
    hidden_act="gelu",
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    initializer_range=0.02,
    layer_norm_eps=1e-12,
)

model = BertForPreTraining(config).to(DEVICE)

The actual training steps are essentially the same as before, because the learning procedure does not depend on how the model was built: a forward pass produces the MLM and NSP logits, the two losses are summed, `loss.backward()` computes the gradients, and the optimizer and scheduler update the weights. Since we deleted the earlier optimizer and scheduler along with the first model, we must create new ones for the new model's parameters, using the same arguments as before: AdamW with selective weight decay, a linear warmup over the first $6\%$ of the steps followed by a linear decay, and the same number of epochs (here: $3$) to compute `total_steps`. Keeping these settings identical also makes the two runs directly comparable, so any difference in the loss curves reflects the model implementation rather than the training setup.

In [39]:
total_steps = EPOCHS * len(train_dl)
optimizer = make_optimizer(model)
scheduler = linear_warmup_decay(optimizer, int(0.06 * total_steps), total_steps)

Overall, the training loop is almost identical. The only minor difference is that the `forward()` method of the `BertForPreTraining` class returns an single output object that contains the MLM logits and the NSP logits as attributes `prediction_logits` and `seq_relationship_logits`. These two attributes we now have to pass to the `compute_loss()` method. Again, depending on the hardware and the availability of a GPU, running the code cell below may take several minutes to complete.

In [40]:
history = dict(step=[], train_mlm=[], train_nsp=[], val_step=[], val_mlm=[], val_nsp_acc=[])
step, t0, run_mlm, run_nsp, run_n = 0, time.time(), 0.0, 0.0, 0
model.train()

for epoch in range(EPOCHS):
    if epoch > 0:
        train_ds.regenerate()                      # fresh sentence pairs each epoch
    for batch in train_dl:
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

        out = model(batch["tok_ids"], batch["seg_ids"], batch["attention_mask"])
        mlm_loss, nsp_loss = compute_loss(out.prediction_logits, out.seq_relationship_logits, batch)
        loss = (mlm_loss + nsp_loss)
        loss.backward()

        run_mlm += mlm_loss.item(); run_nsp += nsp_loss.item(); run_n += 1

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        step += 1

        if step % LOG_EVERY == 0:
            history["step"].append(step)
            history["train_mlm"].append(run_mlm / run_n); history["train_nsp"].append(run_nsp / run_n)
            print(f"Epoch {epoch} step {step}/{total_steps} | mlm {run_mlm/run_n:.3f} | "
                  f"NSP {run_nsp/run_n:.3f} | lr {scheduler.get_last_lr()[0]:.2e} | {time.time()-t0:.0f}s")
            run_mlm = run_nsp = 0.0; run_n = 0

        if step % EVAL_EVERY == 0 or step == total_steps:
            m = evaluate(model, val_dl)
            history["val_step"].append(step); history["val_mlm"].append(m["mlm_loss"])
            history["val_nsp_acc"].append(m["nsp_acc"])
            print(f"  [val] mlm {m['mlm_loss']:.3f} (ppl {m['ppl']:.1f}) | nsp acc {m['nsp_acc']:.3f}")
            torch.save(model.state_dict(), f"{output_folder}bert_mini.pt")

Epoch 0 step 100/1900 | mlm 9.355 | NSP 0.691 | lr 4.39e-04 | 6s
Epoch 0 step 200/1900 | mlm 7.379 | NSP 0.686 | lr 4.76e-04 | 12s
Epoch 0 step 300/1900 | mlm 7.253 | NSP 0.669 | lr 4.48e-04 | 18s
Epoch 1 step 400/1900 | mlm 7.199 | NSP 0.674 | lr 4.20e-04 | 25s
Epoch 1 step 500/1900 | mlm 7.128 | NSP 0.682 | lr 3.92e-04 | 31s
  [val] mlm 7.091 (ppl 1201.7) | nsp acc 0.510
Epoch 1 step 600/1900 | mlm 7.068 | NSP 0.677 | lr 3.64e-04 | 38s
Epoch 1 step 700/1900 | mlm 7.026 | NSP 0.677 | lr 3.36e-04 | 44s
Epoch 2 step 800/1900 | mlm 6.980 | NSP 0.664 | lr 3.08e-04 | 51s
Epoch 2 step 900/1900 | mlm 6.982 | NSP 0.654 | lr 2.80e-04 | 57s
Epoch 2 step 1000/1900 | mlm 6.938 | NSP 0.654 | lr 2.52e-04 | 62s
  [val] mlm 6.942 (ppl 1034.3) | nsp acc 0.534
Epoch 2 step 1100/1900 | mlm 6.922 | NSP 0.652 | lr 2.24e-04 | 69s
Epoch 3 step 1200/1900 | mlm 6.932 | NSP 0.654 | lr 1.96e-04 | 77s
Epoch 3 step 1300/1900 | mlm 6.903 | NSP 0.649 | lr 1.68e-04 | 83s
Epoch 3 step 1400/1900 | mlm 6.891 | NSP 0.65

Unsurprisingly, when we use the same training setup for the manual and the library approach (same data, model size, optimizer, learning rate schedule, and number of epochs), we should see very similar results when training with the `transformers` library. The learning procedure is identical, and only the implementation of the model differs, so the MLM and NSP losses should decrease at a comparable rate, and the validation metrics should end up in a similar range. Small differences are expected, for instance from random initialization, the order of the batches, or minor implementation details such as the initialization scheme, but they should not change the overall picture.

This also means that we should not expect to have learned very meaningful word embeddings yet. The model is tiny, the dataset is small, and the training only lasts a few epochs, so the quality of the embeddings is limited regardless of which implementation we use. As before, the purpose here is not to obtain a strong model, but to verify that the full training pipeline works.

What does differ is the effort. Using the library makes the process much easier to implement: a configuration object, a ready-made model, a data collator, and a trainer replace the many lines of code we wrote by hand, and the result is built on mature, tested, and optimized implementations. At the same time, this convenience hides the core steps of the process, namely how sentence pairs are created, how tokens are masked, how the inputs are embedded, how the two losses are computed, and how the optimization is set up. Having implemented these steps ourselves first makes it much easier to understand what the library does internally, to debug problems when they arise, and to adapt the training to our own needs.

---

## Summary

This notebook went through the core steps of training a BERT model from scratch. We started by turning a raw text corpus (WikiText) into training samples: splitting documents into sentences, building sentence pairs with random negatives for Next Sentence Prediction, adding the special tokens, applying the 80/10/10 corruption for Masked Language Modeling, and creating the token, segment, and position IDs together with the attention mask and labels. We then implemented the model itself, with its embeddings, encoder stack, pooler, MLM and NSP heads, and weight tying, and trained it using a combined cross-entropy loss, AdamW with a warmup and decay schedule, and regular validation. Finally, we saw how the same workflow can be "speed-run" with the `transformers` library, which is more practical but hides most of these details.

The model we trained is tiny and the dataset is small, so we did not expect, and did not obtain, high-quality embeddings. The point was different: by building every component ourselves, we hope the notebook gave you deeper insight into how BERT actually works. Concepts that can seem abstract when described conceptually, such as loss masking, the attention mask for padding, the role of the special tokens, or the interplay of the two pre-training objectives, become concrete once you have seen them as data, tensors, and code. This understanding also helps in practice, for example when debugging a data pipeline, adapting pre-training to a new domain, or interpreting the behavior of a pretrained model.

BERT is worth understanding properly because it is one of the most influential models in modern NLP. It established the pretrain-then-fine-tune paradigm and made contextual embeddings the standard way to represent text. Many later models, such as RoBERTa, ALBERT, DistilBERT, and ELECTRA, are variants of BERT, and its core ideas of self-supervised objectives, special-token input formats, and transfer learning carry over to today's large language models. A solid grasp of how BERT is trained therefore provides a foundation that makes much of the current NLP landscape easier to follow and build upon.

**Disclaimer:** This Jupyter Notebook contains content generated with the assistance of AI. While every effort has been made to review and validate the outputs, users should independently verify critical information before relying on it. The SELENE notebook repository is constantly evolving. We recommend downloading or pulling the latest version of this notebook from Github.